In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
# os.environ["GOOGLE_CSE_ID"] = os.getenv("GOOGLE_CSE_ID")
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")

### __Semantic Search__

In [ ]:
from langchain_core.documents import Document

# Document has three attributes [page_content, metadata, id]
# Document object often represents a chunk of a larger document.

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [2]:
# Get full file name
from pathlib import Path

file_name = "KapilDev_C++Dev_8_Resume_Updated.pdf"
# file_name = "Eb_Notice.pdf"
file_path = Path.cwd() / file_name

print(file_path)


c:\Project\LangChain_Documentation\KapilDev_C++Dev_8_Resume_Updated.pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path)
pdfdocs = loader.load()

In [4]:
print(f"Type of document: {type(pdfdocs)}")
# Printing document attributes [metadata, content, id]
print(f"Document length: {len(pdfdocs)}")

for index, doc in enumerate(pdfdocs):
    print(f"{index}. Document metadata: {doc.metadata['source']}")
    print(f"{index}. Document content: {doc.page_content}") 
    print(f"{index}. Document id: {doc.id}")
    print("-"*100, "\n")

Type of document: <class 'list'>
Document length: 4
0. Document metadata: c:\Project\LangChain_Documentation\KapilDev_C++Dev_8_Resume_Updated.pdf
0. Document content: KAPIL DEV MUTHU 
Senior Software Engineer  
Phone : +91 9629920247 
Email : kapil0707@gmail.com 
 
 
Profile Having 8 years of experience as a Software Engineer in analysis, design, research and  
development of Windows application, CAD addons. Expertise in Product development  
and CAD customization and DevOps activities. Analyze application behavior with 
monitoring tools like Process Monitor, API Monitor, Process explorer and debugging  
tools like Windows Debugger. 
 
Company: SECUDE Solutions India Private Limited, Chennai 
Designation: Senior Software Engineer 
Date: Sep 2022 – July 2025, July 2019 - Mar 2022 
Product: HALOCAD (Data Security) 
• HALOCAD provides Rights Management Solutions offering innovative data 
protection for users of CAD and PLM applications. HALOCAD use DRM solution 
to protect sensitive data.

In [5]:
# Split the documents
#   - LLM Context window limitation
#   - Effective vector embedding
#       - Improved relevance [More precise results. Help in query phase]
#       - Reduced noise [Combining too much information will dilute the real meaning]
#   - Optima RAG
#       - Splitting documents will help us to retrieve only top K relevant chunks


from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    add_start_index=True
)

chunks = text_splitter.split_documents(pdfdocs)

In [14]:
print(f"length of chunks: {len(chunks)}")

for chunk in chunks[:]:
    print(f"Chunk Content:\n")
    print(f"{chunk.page_content}")
    print(f"\nMetadata: {chunk.metadata}")
    print("-"*100, "\n")

length of chunks: 19
Chunk Content:

KAPIL DEV MUTHU 
Senior Software Engineer  
Phone : +91 9629920247 
Email : kapil0707@gmail.com 
 
 
Profile Having 8 years of experience as a Software Engineer in analysis, design, research and  
development of Windows application, CAD addons. Expertise in Product development  
and CAD customization and DevOps activities. Analyze application behavior with 
monitoring tools like Process Monitor, API Monitor, Process explorer and debugging  
tools like Windows Debugger.

Metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-11-23T22:17:10+05:30', 'author': 'Chaya', 'moddate': '2025-11-23T22:17:10+05:30', 'source': 'c:\\Project\\LangChain_Documentation\\KapilDev_C++Dev_8_Resume_Updated.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'start_index': 0}
---------------------------------------------------------------------------------------------------- 

Chunk Content:

tools like Windows Debugger. 

In [8]:
# Create IDs for chunks, in order to keep only unique entries in vector store
# Chroma vector store does not handle duplicate entries

import hashlib

def generate_content_hash(chunk):
    chunk_content = chunk.page_content
    return hashlib.sha256(chunk_content.encode()).hexdigest()

chunk_ids = [generate_content_hash(chunk) for chunk in chunks]
print(f"Length of chunk ids: {len(chunk_ids)}")
print(f"Sample Chunk id: {chunk_ids[0]}")

Length of chunk ids: 19
Sample Chunk id: 8beefbb6a6bac8be09489f3504cc582d4fd368003236d272d324773906634990


In [10]:
# Embeddings [Convert words as vectors]
# This embedding object will be passed as a parameter to vector store constructor
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Simple embedding query
vector = embeddings.embed_query("Hi, how are you?")

print(vector[0:5])

[-0.01811673, -0.0073305904, -0.007106974, -0.052390084, -0.012887501]


In [12]:
# Instanstiate vector store
#   - Store and retrieve documents

from langchain_chroma import Chroma

# Instanstiated vector store
vector_store = Chroma(
    collection_name="Langchain_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

"""
Who converts chunks into vectors?
- The embedding model (the embeddings object).

Who triggers the conversion?
- Chroma, when you call add_documents().

What happens internally:
- You pass text chunks to vector_store.add_documents()
- Chroma calls embeddings.embed_documents(chunks)
- The embedding model converts text → vectors
- Chroma stores those vectors in the vector database
"""

# delete_collection will delete all the chunks from vector store
# vector_store.delete_collection()

# All the chunks are added to vector store
lst_ids = vector_store.add_documents(documents=chunks, ids=chunk_ids)

# print(len(lst_ids))
# print(lst_ids[:3])

# Issue i faced: Multiple times the same chunks is added to vector store
#   - To avoid this, we can use the ids to check if the chunks are already added
#   - If the chunks are already added, then we can skip adding them again
#   - If the chunks are not added, then we can add them


In [13]:
# Different ways of searching in vector store
#   - similarity_search
#   - asimilarity_search
#   - similarity_search_with_score
#   - 

# similarity_search will return list of documents
# List of questions
# "How many years of experience do kapil have?, give me only years of experience"
# "List all the skills of kapil"
# "List all the projects of kapil"
# "List all the companies kapil worked with"

results = vector_store.similarity_search(
    "List all the companies in the document"
)

print(len(results))
for result in results:
    print(result.page_content)
    print("-"*100, "\n")

4
ticket. 
• Interact with PLM team (SIEMENS and SAP) to analyze PLM and CAD integrated  
behavior and further analyze any workflows or requirements 
 
Company: CLOUDIX Global Solutions Private Limited, Chennai 
Date: April 2022 to Aug 2022 
Designation: DevOps Engineer 
 
Project: 
1. Vera (Health care) 
2. Mediguru (Health care) 
3. Palo-Alto (Data Analysis) 
 
 
 
 
2
---------------------------------------------------------------------------------------------------- 

to reduced DevOps team dependencies for household activities like getting logs, 
monitor status of apps in one dashboard. 
• Plan and setup environment for Palo-Alto team to help in development process 
and other DevOps activities. 
 
 
Company: P3 DESIGN SOLUTIONS PVT LTD, Chennai 
On Site: Mahindra Research Valley, Chennai 
Date: June 2017 to July 2019 
Designation: Software Engineer 
 
Project: 
1. NX addon for Configurable design platform (Mahindra and Swaraj)
------------------------------------------------------

In [ ]:
# Get similarity search results asychronously
results = await vector_store.asimilarity_search("List all the companies name in the document")

print(len(results))
for result in results:
    print(result.page_content)
    print("-"*100, "\n")

In [ ]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("List all the companies names")

print(f"length of result: {len(results)}\n\n")

for result in results:
    doc, score = result
    print(f"Score: {score}\n")
    print(doc.page_content)
    print("-"*100, "\n")

In [ ]:
embedding = embeddings.embed_query("List all the companies kapil worked for")

results = vector_store.similarity_search_by_vector(embedding)

print(f"length of result: {len(results)}\n\n")

for result in results:
    print(result.page_content)
    print("-"*100, "\n")

In [ ]:
# LangChain VectorStore objects do not subclass Runnable. 
# LangChain Retrievers are Runnables, so they implement a standard set of methods 
#       (e.g., synchronous and asynchronous invoke and batch operations). 
# Although we can construct retrievers from vector stores, retrievers can interface with 
#       non-vector store sources of data, as well (such as external APIs).

# "How many years of experience do kapil have?, give me only years of experience"
# "List all the skills of kapil"
# "List all the projects of kapil"
# "List all the companies kapil worked with"

from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)

lst_query =     [
        "Give me the total years of experience",
        "List his skills",
        "Give me the project information he worked",
        "List all the companie names"
    ]

results = retriever.batch(
    lst_query,
)

for index, result in enumerate(results, start=1):
    print(f"{index}. {lst_query[index-1]}\n")

    for doc in result:
        print(doc.page_content)
        print("-"*100, "\n")

    print("#"*100, "\n")


In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "Give me the total years of experience",
        "List his skills",
    ],
)

### __RAG__

#### __Search Website Content__

In [15]:
import bs4
from langchain.agents import AgentState, create_agent
from langchain_community.document_loaders import WebBaseLoader
from langchain.messages import MessageLikeRepresentation
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load and chunk contents of the blog
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

print(f"Length of documents: {len(docs)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Length of documents: 1


In [19]:
# print(docs[0].page_content)

In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
print(f"Length of chunks: {len(all_splits)}")

Length of chunks: 63


In [31]:
web_chunk_ids = [generate_content_hash(chunk) for chunk in all_splits]
print(f"Length of chunk_id: {len(web_chunk_ids)}")
print(f"Sample chunk id: {web_chunk_ids[0]}")

Length of chunk_id: 63
Sample chunk id: 24dc4270a399ca9dbc2880a3721fb41b614e29c2f8555cb66bc630d40ece4c16


In [32]:
# Create embeddings
web_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Create individual vector store for web search
web_vector_store = Chroma(
    embedding_function=web_embeddings,
    persist_directory="./web_chroma_db",
)


# Index chunks
_ = web_vector_store.add_documents(documents=all_splits, ids= web_chunk_ids)

In [35]:
from langchain.tools import tool

# Construct a tool for retrieving context
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = web_vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

tools = [retrieve_context]

In [36]:
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)

In [37]:
# Two ways to create agent
# Type: 1
agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=tools,
    system_prompt=prompt
    )

In [ ]:
# # Type: 2. This will add more control over model confguration
# from langchain_google_genai import ChatGoogleGenerativeAI

# # Docs: https://reference.langchain.com/python/integrations/langchain_google_genai/ChatGoogleGenerativeAI/?h=chatgooglegenerativeai
# gemini_model = ChatGoogleGenerativeAI(
#     model="google_genai:gemini-3-flash-preview",
#     system_prompt=prompt,
#     tools=tools
# )

In [ ]:
# Getting response using stream method
query = "What is task decomposition?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is task decomposition?
================================== Ai Message ==================================

[]
Tool Calls:
  retrieve_context (4fb3f5e0-b530-43d2-a85a-0ea0e1cd9350)
 Call ID: 4fb3f5e0-b530-43d2-a85a-0ea0e1cd9350
  Args:
    query: What is task decomposition?
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language

In [45]:
# Getting response using invoke method
query = "What is task decomposition?"

web_response = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

print(web_response)
print(type(web_response))

web_response["messages"][-1].pretty_print()

# for message in web_response:
#     print(f"Message Content: {message['content']}")
#     print(f"\nMessage Metadata: {message['metadata']}")
#     print('-'*100, "\n")
    

{'messages': [HumanMessage(content='What is task decomposition?', additional_kwargs={}, response_metadata={}, id='a4e8eaca-542d-4346-8123-da945e57c47b'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'retrieve_context', 'arguments': '{"query": "What is task decomposition?"}'}, '__gemini_function_call_thought_signatures__': {'3e2561bd-a54e-4461-a6f5-005e574a1cf6': 'EpECCo4CAXLI2nxu+8RnISQfrwSliS425mhIi1igjufKkQD+IbPJDXUP/F/bGIk0Ctl87dltHQgBQrUMQgEcMOQtCIuUgL24I3jUh8ZbX3GRcTuWNphx7AI4noRdIV+zLmvtcopaGFYlCGSrr7799bTajJeRHMdnvpC5EeCDGaTX5PNqfkbZ9fWD2VV7xs+05Wq45dR9bypqzoZkq0VvxroXn+sQytLruuvdQaTSSzrcvOyr+yccEfUpLU7g4oI+KrEw2sJW+5TwTCfDcjn0f3y4N2hNOVJITrbOCvKXOUb8dM20bf9nEnFwr7+nX2vvT9WYFbSPls9LF7ayMIGu0PPbsFBoy5Av92BKiTXC4k8ECFOJ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bb1bb-b8ca-7392-b592-a13e5f518414-0', tool_calls=[{'name': 'retrieve_context', 'arg

In [46]:
web_response["messages"][0].pretty_print()

================================ Human Message =================================

What is task decomposition?


In [47]:
web_response["messages"][1].pretty_print()

================================== Ai Message ==================================

[]
Tool Calls:
  retrieve_context (3e2561bd-a54e-4461-a6f5-005e574a1cf6)
 Call ID: 3e2561bd-a54e-4461-a6f5-005e574a1cf6
  Args:
    query: What is task decomposition?


In [48]:
web_response["messages"][2].pretty_print()

================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an e

In [49]:
web_response["messages"][3].pretty_print()

================================== Ai Message ==================================

[{'type': 'text', 'text': 'Task decomposition is a technique used to break down a complex task into smaller, simpler, and more manageable steps. This process allows an AI agent or model to handle complicated tasks more effectively by planning ahead and addressing each sub-task sequentially or hierarchically.\n\nAccording to the blog post, task decomposition can be achieved through several methods:\n\n*   **LLM Prompting:** Using simple prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?" to guide the model.\n*   **Task-Specific Instructions:** Providing specific directions tailored to the task, such as "Write a story outline" for the larger task of writing a novel.\n*   **Human Input:** Incorporating direct guidance from a human user.\n*   **Chain of Thought (CoT):** A standard technique where the model is instructed to "think step by step," transforming a big task into multiple mana

In [52]:
# The best way to print the final response from LLM is to indes the response using [-1]
# This is because, there may be n number of tool calls, some times no tool call is there
# To always access last message, use [-1] as indexing
web_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

[{'type': 'text', 'text': 'Task decomposition is a technique used to break down a complex task into smaller, simpler, and more manageable steps. This process allows an AI agent or model to handle complicated tasks more effectively by planning ahead and addressing each sub-task sequentially or hierarchically.\n\nAccording to the blog post, task decomposition can be achieved through several methods:\n\n*   **LLM Prompting:** Using simple prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?" to guide the model.\n*   **Task-Specific Instructions:** Providing specific directions tailored to the task, such as "Write a story outline" for the larger task of writing a novel.\n*   **Human Input:** Incorporating direct guidance from a human user.\n*   **Chain of Thought (CoT):** A standard technique where the model is instructed to "think step by step," transforming a big task into multiple mana